In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the dataset we engineered yesterday
df = pd.read_csv('../data/processed/erlangen_features.csv')
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# 2. Chronological Splitting to prevent data leakage
train_df = df.loc['2010':'2021']
val_df   = df.loc['2022':'2022']
test_df  = df.loc['2023':'2024']

print(f"📅 Train rows (2010-2021): {len(train_df)}")
print(f"📅 Val rows   (2022):      {len(val_df)}")
print(f"📅 Test rows  (2023-2024): {len(test_df)}")

📅 Train rows (2010-2021): 4353
📅 Val rows   (2022):      365
📅 Test rows  (2023-2024): 731


In [2]:
# The actual irradiance value we want to predict
y_test = test_df['GHI']

# The Naive Baseline simply guesses yesterday's value (lag_1)
y_pred_naive = test_df['ghi_lag_1']

# Calculate evaluation metrics
mae_naive = mean_absolute_error(y_test, y_pred_naive)
rmse_naive = np.sqrt(mean_squared_error(y_test, y_pred_naive))
r2_naive = r2_score(y_test, y_pred_naive)

print("🏁 --- Naive Baseline (Lag-1) Results on Test Set ---")
print(f"🔹 MAE:  {mae_naive:.2f} W/m²")
print(f"🔹 RMSE: {rmse_naive:.2f} W/m²")
print(f"🔹 R² Score: {r2_naive:.4f}")

🏁 --- Naive Baseline (Lag-1) Results on Test Set ---
🔹 MAE:  0.84 W/m²
🔹 RMSE: 1.18 W/m²
🔹 R² Score: 0.6961


In [5]:
# 1. List the exact columns we want the ML models to look at
feature_cols = [
    'T2M', 'CLOUD_AMT', 'WS10M', 
    'solar_elevation', 'solar_zenith',
    'month', 'day_of_year',
    'ghi_lag_1', 'ghi_lag_7', 'ghi_lag_30',
    'ghi_rolling_mean_7', 'ghi_rolling_std_7'
]

# 2. Separate into X (features) and y (target) matrices
X_train, y_train = train_df[feature_cols], train_df['GHI']
X_val, y_val     = val_df[feature_cols], val_df['GHI']
X_test, y_test   = test_df[feature_cols], test_df['GHI']

print(f"🌲 Training feature shape: {X_train.shape}")
print(f"🎯 Training target shape:  {y_train.shape}")

🌲 Training feature shape: (4353, 12)
🎯 Training target shape:  (4353,)


In [6]:
from sklearn.ensemble import RandomForestRegressor

print("🏋️ Training Random Forest Regressor (this may take a few seconds)...")

# Initialize the model
# random_state ensures that your results stay perfectly reproducible
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Train the model on our historical data
rf_model.fit(X_train, y_train)

# Predict on the unseen Test Set
y_pred_rf = rf_model.predict(X_test)

# Calculate evaluation metrics for Random Forest
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

# Calculate percentage improvement over Naive baseline
pct_improvement = ((mae_naive - mae_rf) / mae_naive) * 100

print("\n🌲 --- Random Forest Results on Test Set ---")
print(f"🔸 MAE:  {mae_rf:.2f} W/m²")
print(f"🔸 RMSE: {rmse_rf:.2f} W/m²")
print(f"🔸 R² Score: {r2_rf:.4f}")
print(f"🚀 Improvement over Baseline: {pct_improvement:.2f}% better MAE!")

🏋️ Training Random Forest Regressor (this may take a few seconds)...

🌲 --- Random Forest Results on Test Set ---
🔸 MAE:  0.42 W/m²
🔸 RMSE: 0.60 W/m²
🔸 R² Score: 0.9221
🚀 Improvement over Baseline: 50.05% better MAE!
